# LSEG Data libraries API

**Natural Language Processing for News**

Dr. Yves J. Hilpisch | The Python Quants GmbH

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>

<img src="http://hilpisch.com/images/tr_eikon_02.png" width=350px align=left>

## The Agenda

This tutorial covers **natural language processing (NLP)** based on news from the Eikon Data API:

* Retrieving News Headlines and Story Texts
* Extracting Raw Text from HTML
* Tokenizing a Raw Text
* Collecting Raw Texts and Tokenizing Them
* Building a Vocabulary for a Raw Text Collection

## Imports and Versions

The following imports several **packages** as used in the following.

In [2]:
import nltk, bs4  # NLP toolkit & BeautyfulSoup
import lseg.data as ld  # the LSEG Data libraries
from bs4 import BeautifulSoup  # HTML parsing
from nltk import word_tokenize  # tokenizing
import configparser as cp

The following **Python and package versions** are used.

In [3]:
import sys
print(sys.version)

3.9.7 (default, Sep 16 2021, 16:59:28) [MSC v.1916 64 bit (AMD64)]


In [4]:
ld.__version__

'2.0.0'

In [5]:
nltk.__version__

'3.7'

In [6]:
bs4.__version__

'4.12.3'

## Connecting to LSEG Data Libraries API

In [7]:
ld.open_session()

<lseg.data.session.Definition object at 0x15e456eb550 {name='workspace'}>

## Reading News Headlines

The function `ld.news.get_headlines()` allows you to search for and retrieve **news headlines**, including `storyId` values needed to retrieve the full news text.

A `query` string might contain `RICs` and other words to be searched for.

In [8]:
news = ld.news.get_headlines('R:TSLA.O PRODUCTION',
                         start='2024-02-15',
                         end='2024-03-16',
                         count=20
                        )

In [9]:
news

,headline,storyId,sourceCode
versionCreated,,,
2024-03-15 16:06:40.909,Tesla Model 2 set for production next year as ...,urn:link:webnews:20240315:nNRAs369cm:0,NS:AUTOCA
2024-03-15 13:05:39.309,"Tesla's Cybertruck Production Surges, Defying ...",urn:link:webnews:20240315:nNRAs3402s:0,NS:WEBPRO
2024-03-15 06:12:41.280,Tesla Model 2 set for production later this ye...,urn:link:webnews:20240315:nNRAs2zcm8:0,NS:AUTOCA
2024-03-15 01:20:12.658,Musk Talks Tesla Semi & Lower Priced Car Produ...,urn:link:webnews:20240315:nNRAs2wzm8:0,NS:CLECON
2024-03-14 07:25:07.275,Musk visits German Tesla plant as production r...,urn:newsml:newsroom:20240314:nNRAs2lt8f:0,NS:WASTIM
2024-03-14 06:16:49.260,'We Are Back:' Tesla CEO Elon Musk Visits Berl...,urn:newsml:newsroom:20240314:nNRAs2l3y7:0,NS:BENZIN
2024-03-13 19:21:58.793,Tesla CEO Elon Musk visits Giga Berlin as prod...,urn:link:webnews:20240313:nNRAs2fkhz:0,NS:YAHNEX
2024-03-13 17:03:06.300,Elon Musk carries his son X AE A-XIII as he vi...,urn:newsml:newsroom:20240313:nNRAs2cd95:0,NS:DAIONL
2024-03-13 14:32:17.251,Elon Musk visits a Tesla plant near Berlin as ...,urn:newsml:newsroom:20240313:nNRAs2bkr1:0,NS:DETNEW


## Retrieving Full Text

The function `ld.news.get_story()` retrieves the full text of a **news story** given the `storyId` value.

The `storyId` values are stored in the respective column of the `news` `DataFrame` object as created above.

In [10]:
baseurl = "/data/news/v1/stories/"

In [11]:
news['storyId']

versionCreated
2024-03-15 16:06:40.909           urn:link:webnews:20240315:nNRAs369cm:0
2024-03-15 13:05:39.309           urn:link:webnews:20240315:nNRAs3402s:0
2024-03-15 06:12:41.280           urn:link:webnews:20240315:nNRAs2zcm8:0
2024-03-15 01:20:12.658           urn:link:webnews:20240315:nNRAs2wzm8:0
2024-03-14 07:25:07.275        urn:newsml:newsroom:20240314:nNRAs2lt8f:0
2024-03-14 06:16:49.260        urn:newsml:newsroom:20240314:nNRAs2l3y7:0
2024-03-13 19:21:58.793           urn:link:webnews:20240313:nNRAs2fkhz:0
2024-03-13 17:03:06.300        urn:newsml:newsroom:20240313:nNRAs2cd95:0
2024-03-13 14:32:17.251        urn:newsml:newsroom:20240313:nNRAs2bkr1:0
2024-03-13 13:55:38.483        urn:newsml:newsroom:20240313:nNRAs291qp:0
2024-03-13 13:55:07.989    urn:newsml:reuters.com:20240313:nNRAs293ep:14
2024-03-13 12:03:47.928     urn:newsml:reuters.com:20240313:nMnw168982:1
2024-03-13 07:47:14.484           urn:link:webnews:20240313:nNRAs26rr4:0
2024-03-13 07:32:47.000     urn:news

To read a news story, **pick out one story via its storyId and display the full text** provided as HTML code.

In [12]:
storyId = 'urn:newsml:reuters.com:20240313:nNRAs293ep:14'

In [13]:
request_definition = ld.delivery.endpoint_request.Definition(
        url = baseurl + storyId,
        method = ld.delivery.endpoint_request.RequestMethod.GET
    )
response = request_definition.get_data()

In [14]:
rawr = response.data.raw
if 'newsItem' in rawr.keys():
    storyText = rawr['newsItem']['contentSet']['inlineData']['$']

In [15]:
from IPython.display import HTML

In [16]:
HTML(storyText)

## Extracting Raw Text

For the purposes of parsing the story text, **raw text** is better suited than HTML. To this end, the `bs4` package is helpful in transforming the HTML content to text.

In [17]:
ind = storyText.find('Tesla')

In [18]:
print(storyText[ind:ind + 500])

Tesla CEO Elon Musk visited the electric car
maker's first European plant on Wednesday as production resumed at the factory
just outside Berlin, about a week after a suspected arson attack
(https://apnews.com/article/germany-tesla-factory-berlin-arson-9d66c3efa104d03244bd2c4d920c35a0)
cut its power supply.

Musk was expected at a "team huddle" with employees at the plant in the
Gruenheide municipality, employee council chief Michaela Schmitz told regional
broadcaster RBB's Inforadio channel. Rep


## Tokens for a Text

Using `nltk` **tokenization**, i.e., the splitting up of a raw text into unique elements, is easily accomplished. The `punkt` package for `nltk` is needed.

In [19]:
nltk.download('punkt')  # downloads package if required

[nltk_data] Downloading package punkt to C:\Users\Marios
[nltk_data]     Skevofylakas\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [20]:
tokens = word_tokenize(storyText)  # derives tokens for the raw text

In [21]:
tokens[20:40]

['newsml',
 ':',
 'reuters.com:20240313',
 ':',
 'nNRAs293ep',
 '&',
 'default-theme=true',
 'GRUENHEIDE',
 ',',
 'Germany',
 '(',
 'AP',
 ')',
 '—',
 'Tesla',
 'CEO',
 'Elon',
 'Musk',
 'visited',
 'the']

On the basis of tokens, **contexts** for different tokens (words) are easily selected.

In [22]:
text = nltk.Text(tokens)

In [23]:
text.concordance('production')

Displaying 2 of 2 matches:
st European plant on Wednesday as production resumed at the factory just outsi
rocks '' or `` Germany rocks . '' Production at Tesla 's plant in Gruenheide c


## Collecting Raw Texts

The analyses that follow are based on **all news stories** as seen above. To this end, the raw texts are collected in a `list` object.

In [24]:
stories = []
for storyId in news['storyId']:
    request_definition = ld.delivery.endpoint_request.Definition(
        url = baseurl + storyId,
        method = ld.delivery.endpoint_request.RequestMethod.GET
    )
    response = request_definition.get_data()

    rawr = response.data.raw
    if 'newsItem' in rawr.keys():
        storyText = rawr['newsItem']['contentSet']['inlineData']['$']
        stories.append(storyText)

In [25]:
for story in stories:
    print(story[120:200])

t Monday after arsonists burned a nearby electric pylon March 5. According to Te
(NASDAQ:TSLA) CEO Elon Musk on Wednesday visited the company's gigafactory in Be
d production to stop, costing hundreds of millions of dollarsElon Musk carries h
la CEO Elon Musk visited the electric car maker's first European plant on Wednes
— Tesla CEO Elon Musk visited the electric car maker's first European plant on W
v1/story?guid=urn:newsml:reuters.com:20240313:nNRAs293ep&default-theme=true


GR
ction 
resumed at the factory just outside Berlin about a week after 
a suspecte
ERLIN, 13 mars (Reuters) - La production a repris mercredi à
la gigafactory euro
many, local radio broadcaster rbb reported, after an arson
attack on a nearby el
many, local radio broadcaster rbb reported, after an arson
attack on a nearby el
ory 
 • Elon Musk arrived at Tesla's European gigafactory in Gruenheide, Germany
ews agency Bernama on its website:

The US electric car manufacturer Tesla said 


## Tokens for Raw Texts

The same approach is now applied to all the story texts as collected above.

In [26]:
collection = ''.join(stories)  # combines all texts

In [27]:
tokens = word_tokenize(collection)  # derives tokens from the collection

In [28]:
tokens[40:60]

['has',
 'been',
 'restarted',
 'and',
 'primed',
 'for',
 'production',
 '.',
 'Tesla',
 'says',
 'the',
 'unexpected',
 'work',
 'stoppage',
 'likely',
 'cost',
 'the',
 'company',
 'millions',
 'in']

Based on the new set of tokens, the collection of raw texts can now be searched and contexts can be looked up.

In [29]:
text = nltk.Text(tokens)

In [30]:
text.concordance('production')

Displaying 25 of 36 matches:
Musk visits German Tesla plant as production resumes after eco terror attackPo
has been restarted and primed for production . Tesla says the unexpected work 
, one week after the plant halted production following an arson attack by radi
has been restarted and primed for production . Tesla says the unexpected work 
ts Berlin Gigafactory With Son As Production Resumes After Arson AttackTesla I
his son after the factory resumed production following a week-long pause cause
tion from a darkened factory on a production pause to an actively manufacturin
lost power and was forced to halt production on Tuesday , March . 5 , followin
o double its capacity for battery production to 100 gigawatt hours and car pro
ion to 100 gigawatt hours and car production to 1 million units annually . Che
-terrorist ' arsonists who forced production to stop , costing hundreds of mil
-terrorist ' arsonists who forced production to stop , costing hundreds of mil
fter the plant was forc

In [31]:
text.concordance('increase')

Displaying 1 of 1 matches:
 damage to Tesla is also likely to increase . Earlier , the company cited dama


In [32]:
text.concordance('automation')

no matches


In [33]:
text.concordance('vehicles')

Displaying 3 of 3 matches:
ehicle capacity of 375,000 Model Y vehicles . The company is currently looking
st production by up to one million vehicles annually to feed Europe 's growing
ifting away from combustion engine vehicles . But the plans have annoyed local


## Building a Vocabulary

Based on the tokens, a **vocabulary** can be created.

In [34]:
words = sorted([w.lower() for w in tokens])

In [35]:
ind = words.index('a')  # first occurance of 'a'
ind

588

In [36]:
words[ind: ind+15]

['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']

The following code **deletes duplicates** and **sorts** the remaining `list` object alphabetically.

In [37]:
words = sorted(list(set(words[ind:])))

In [38]:
words[:20]

['a',
 'a-xiii',
 'able',
 'about',
 'absolutely',
 'access',
 'accompanied',
 'according',
 'accused',
 'accusing',
 'acres',
 'actions',
 'actively',
 'activist',
 'activists',
 'activités',
 'add',
 'added',
 'addressing',
 'adjoined']

## Conclusions

This tutorial covers the following **natural language processing (NLP)** tasks based on the LSEG Data Libraries API and respective Python packages:

* Retrieving News Headlines and Story Texts
* Extracting Raw Text from HTML
* Tokenizing a Raw Text
* Collecting Raw Texts and Tokenizing Them
* Building a Vocabulary for a Raw Text Collection

## LSEG Data Libraries API Developer Resources

* [API Pages](https://developers.lseg.com/en/api-catalog) 
* [Q&A Forums](https://community.developers.refinitiv.com/index.html) 

Data Item Browser Application: Type `DIB` into Workspace Search Bar.

* [Article on Chains](https://developers.lseg.com/en/article-catalog/article/simple-chain-objects-ema-part-1)